### Next Word Generator

In [1]:
sentences = [
    "I am learning machine learning",
    "I like deep learning models",
    "I am playing with friends",
    "I am reading a book",
    "I am studying mathematics",
    "I am working on a project",
    "machine learning is fun to learn",
    "deep learning uses neural networks",
    "natural language processing is interesting",
    "python is widely used in data science",
    "data science includes machine learning",
    "students are studying artificial intelligence",
    "models learn patterns from data",
    "neural networks have many layers",
    "training data is important for models",
    "testing helps evaluate performance",
    "optimization improves model accuracy",
    "algorithms solve complex problems",
    "feature engineering improves results",
    "big data requires efficient processing",
    "cloud computing supports scalability",
    "software development requires practice",
    "coding improves problem solving skills",
    "debugging helps fix errors",
    "practice makes a person better"
]

Tokenization

In [2]:
tokenized_sentences = [sentence.split() for sentence in sentences]

print(tokenized_sentences[:3])

[['I', 'am', 'learning', 'machine', 'learning'], ['I', 'like', 'deep', 'learning', 'models'], ['I', 'am', 'playing', 'with', 'friends']]


Vocabulary

In [3]:
from collections import Counter

words = [word for sentence in tokenized_sentences for word in sentence]
vocab = sorted(set(words))

word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

print("Vocabulary size:", len(vocab))
print("Sample vocab:", vocab[:10])

Vocabulary size: 83
Sample vocab: ['I', 'a', 'accuracy', 'algorithms', 'am', 'are', 'artificial', 'better', 'big', 'book']


Context Window and Target generation

In [4]:
context_size = 2

contexts = []
targets = []

for sentence in tokenized_sentences:
    for i in range(len(sentence) - context_size):
        context = sentence[i:i+context_size]
        target = sentence[i+context_size]

        contexts.append(context)
        targets.append(target)

Sliding Window

In [5]:
for i in range(10):
    print(f"{contexts[i]} ---> {targets[i]}")

['I', 'am'] ---> learning
['am', 'learning'] ---> machine
['learning', 'machine'] ---> learning
['I', 'like'] ---> deep
['like', 'deep'] ---> learning
['deep', 'learning'] ---> models
['I', 'am'] ---> playing
['am', 'playing'] ---> with
['playing', 'with'] ---> friends
['I', 'am'] ---> reading


Text to Index 

In [6]:
import numpy as np

X = [[word_to_index[word] for word in context] for context in contexts]
y = [word_to_index[word] for word in targets]

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (72, 2)
y shape: (72,)


Index to Embedding vectors

In [7]:
import torch
import torch.nn as nn

class NextWordModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(context_size * embedding_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.shape[0], -1)
        out = self.fc(x)
        return out

Training

In [9]:
vocab_size = len(vocab)
embedding_dim = 10

model = NextWordModel(vocab_size, embedding_dim, context_size)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# FIX HERE 👇
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = loss_fn(outputs, y_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 4.623752593994141
Epoch 10, Loss: 3.071373224258423
Epoch 20, Loss: 1.8002063035964966
Epoch 30, Loss: 0.9206774830818176
Epoch 40, Loss: 0.4691782593727112
Epoch 50, Loss: 0.28332090377807617
Epoch 60, Loss: 0.2128925919532776
Epoch 70, Loss: 0.18323472142219543
Epoch 80, Loss: 0.16850349307060242
Epoch 90, Loss: 0.16009755432605743


Prediction of next word

In [10]:
def predict_next(context_words):
    context_idx = torch.tensor([[word_to_index[word] for word in context_words]])
    output = model(context_idx)
    predicted_idx = torch.argmax(output).item()
    return index_to_word[predicted_idx]

Results

In [13]:
print(predict_next(["I", "am"]))
print(predict_next(["deep", "learning"]))
print(predict_next(["machine", "learning"]))

reading
uses
is
